# CD Atlas: h5ad Conversion and Metadata Assembly

Converts the merged CD atlas from a dense count matrix (exported from R)
to an AnnData `.h5ad` file, then attaches final cell type annotations.

**Inputs** (exported from `07_cd_atlas_assembly.R`):
- `full_9p_integ_p19_upd_anno.csv.gz`        – merged count matrix (long format)
- `full_meta_9p_integ_p19_upd_anno.csv`       – cell metadata before final anno
- `meta_cleaned_annoed_all_cell_types.csv`    – final per-cell annotations
- `cell_type_category_map.csv`               – maps cell_type → category/short

**Output**:
- `cleaned_annoed_all_cell_types.h5ad`        – annotated AnnData object

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix
import scanpy as sc
import gc

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cd/scrna/output"
MERGED_DIR = f"{DATA_DIR}/merged_cd"

## 1. Load count matrix and build sparse AnnData

In [ ]:
# Load long-format count matrix (gene, cell, count)
df = pd.read_csv(f"{MERGED_DIR}/full_9p_integ_p19_upd_anno.csv.gz", compression="gzip")
print(f"Loaded {len(df):,} non-zero entries")

In [ ]:
# Build cell × gene sparse matrix
n_cells = df['cell'].nunique()
n_genes = df['gene'].nunique()
print(f"Unique cells: {n_cells:,}, unique genes: {n_genes:,}")

cell_cat = pd.Categorical(df['cell'])
gene_cat = pd.Categorical(df['gene'])
mat = coo_matrix(
    (df['count'].to_numpy(), (cell_cat.codes, gene_cat.codes)),
    shape=(n_cells, n_genes)
).tocsr()

cells = cell_cat.categories.to_numpy()
genes = gene_cat.categories.to_numpy()
del df; gc.collect()

In [ ]:
# Load final annotations (subset of cells that passed QC + annotation)
meta2  = pd.read_csv(f"{MERGED_DIR}/meta_cleaned_annoed_all_cell_types.csv")
mapper = pd.read_csv(f"{MERGED_DIR}/cell_type_category_map.csv")

# Attach cell category and short labels from mapper
meta2 = meta2.merge(
    mapper[['cell type', 'cell type general', 'cell category', 'cell type short']],
    how='left', left_on='label', right_on='cell type'
)

In [ ]:
# Filter count matrix to annotated cells only
keep_cells = meta2['Unnamed: 0'].tolist()
cell_idx   = pd.Series(range(len(cells)), index=cells)
keep_idx   = cell_idx[cell_idx.index.isin(keep_cells)].values

mat_filt = mat[keep_idx, :]

# Drop non-gene features (microRNAs, pseudogenes)
drop_features = ['7SK-ENSG00000260682', 'bP-21264C1.2',
                 'hsa-mir-1199', 'hsa-mir-150', 'hsa-mir-335', 'hsa-mir-8072']
gene_mask = ~pd.Series(genes).isin(drop_features).values
mat_filt  = mat_filt[:, gene_mask]
genes_filt = genes[gene_mask]

print(f"Filtered matrix: {mat_filt.shape[0]:,} cells × {mat_filt.shape[1]:,} genes")

## 2. Build AnnData and write h5ad

In [ ]:
adata = sc.AnnData(
    X   = csr_matrix(mat_filt.astype(np.float32)),
    obs = meta2.copy(),
    var = pd.DataFrame(index=genes_filt)
)
adata.obs_names = adata.obs['Unnamed: 0'].tolist()
adata.obs       = adata.obs.drop(['Unnamed: 0'], axis=1)

print(adata)

In [ ]:
adata.write(f"{MERGED_DIR}/cleaned_annoed_all_cell_types.h5ad")
print("Written:", f"{MERGED_DIR}/cleaned_annoed_all_cell_types.h5ad")

## 3. Validate and inspect

In [ ]:
adata = sc.read_h5ad(f"{MERGED_DIR}/cleaned_annoed_all_cell_types.h5ad")
print(adata)
print("\nCell type counts:")
print(adata.obs['label'].value_counts())

In [ ]:
print("Condition distribution:")
print(adata.obs['condition'].value_counts())
print("\nDataset distribution:")
print(adata.obs['source'].value_counts())